<a href="https://colab.research.google.com/github/nikitask14/uci-bank-marketing-study/blob/main/NN_Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**uci-bank-marketing-study**

**Step 1: Load the Python Libraries**

In [2]:
import pandas as pd
import numpy as np

**Step 2: Load the dataset**

In [4]:
# df = pd.read_csv("bank-additional-full.csv")
df = pd.read_csv("bank-additional-full.csv", sep = ";")

After inspecting the raw file we have discovered that data in the following format:

"value";"value";"value"

This causes pandas to never find the comma seperator it has been looking for and treats an entire physical row as one field.

**So, we have to tell the pd.read_csv() that delimiter in this file is ; and not ,**

So we use:
df = pd.read_csv("bank-additional-full.csv", sep = ";") instead of

df = pd.read_csv("bank-additional-full.csv")


In [5]:
df.shape

(41188, 21)

In [6]:
df.head()


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


#### **Step 3: UCI Bank Marketing Data Profiling**
We are using the **UCI Bank Marketing dataset**, specifically:


*bank-additional-full.csv*


The dataset contains:


- 41,188 observations
- 21 columns


These consist of:

- 20 input variables
- 1 binary target variable, ***y***

The target indicates whether the client subscribed to a **term deposit**: yes / no

---

#### **Understanding the information available in the dataset**

The input variables can be grouped conceptually according to what kind of information they represent.

##### **1. Client information**

Examples include:


- age
- job
- marital
- education
- default
- housing
- loan


These variables describe the client and would generally be known before the current marketing interaction takes place.

---

##### **2. Previous campaign information**


- pdays
- previous
- poutcome


These variables describe the client's history from previous marketing campaigns.

For some clients, previous campaign information exists; for others, there was no previous contact.

This information would also be available before the current campaign interaction begins.

---

##### **3. Wider economic information**


- emp.var.rate
- cons.price.idx
- cons.conf.idx
- euribor3m
- nr.employed


These variables describe the wider economic environment rather than the individual client.

They may be available before the current interaction, depending on the exact prediction point.

---

##### **4. Current campaign/contact information**

Examples include:


- contact
- month
- day_of_week
- campaign
- duration


These variables are different because their availability depends on when the prediction is being made.

Some may be known once the current campaign is underway, while others, such as **duration**, can only be known after the contact itself has taken place.

This led to the key research-design insight:

> **The validity of a feature depends not only on whether it exists in the dataset, but also on whether it would actually be available at the moment the prediction is supposed to be made.**

---


 **Step 4: Research Problem Formulation**

 Research problem
Instead of treating the dataset simply as:

- **features → predict subscription to term deposit**

the study will compare two different information states.

---
**Problem A — Pre-campaign / pre-contact prediction**

Use only information that would be available before the current campaign interaction:

```
client information
+
previous campaign history
+
wider economic information
```
**Research question:**


How well can we predict whether a client will subscribe to a term deposit using only information available before the current campaign interaction?

---

**Problem B — Interaction-informed prediction**
Use the same pre-campaign information and add information generated during the current campaign/contact.

**Research question:**

How much does predictive performance change when current-campaign information is added?

**Main research question**

```Does adding current-campaign information improve our ability to predict the final term-deposit subscription outcome compared with using only information available before the current campaign interaction?```

This creates a controlled comparison between:

- pre-campaign information

vs
- pre-campaign + current-campaign information

The aim is to measure whether current-campaign information provides additional predictive signal.

**Why this project adds value**

This project is not simply another binary-classification exercise.

It introduces a new layer of research judgment by requiring us to reason about:

- what real-world decision the model is supposed to support,
- when the prediction is made,
- which information is legitimately available at that moment,
- how to design a fair controlled comparison,
- and what conclusions the experiment can and cannot support.
- At the same time, it extends the technical workflow by moving from a single-model experiment toward a controlled, repeatable comparison between two experimental conditions.


**Step 5: Inspecting Data types**

In [8]:
# info only tells us that pandas didn't recognise any entry as NaN
# but it could use ?, unknown or any other numerical identifier for
# missingness
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [10]:
df["age"].unique()

array([56, 57, 37, 40, 45, 59, 41, 24, 25, 29, 35, 54, 46, 50, 39, 30, 55,
       49, 34, 52, 58, 32, 38, 44, 42, 60, 53, 47, 51, 48, 33, 31, 43, 36,
       28, 27, 26, 22, 23, 20, 21, 61, 19, 18, 70, 66, 76, 67, 73, 88, 95,
       77, 68, 75, 63, 80, 62, 65, 72, 82, 64, 71, 69, 78, 85, 79, 83, 81,
       74, 17, 87, 91, 86, 98, 94, 84, 92, 89])

In [13]:
for column in df.columns:
  print(column, df[column].unique())

age [56 57 37 40 45 59 41 24 25 29 35 54 46 50 39 30 55 49 34 52 58 32 38 44
 42 60 53 47 51 48 33 31 43 36 28 27 26 22 23 20 21 61 19 18 70 66 76 67
 73 88 95 77 68 75 63 80 62 65 72 82 64 71 69 78 85 79 83 81 74 17 87 91
 86 98 94 84 92 89]
job ['housemaid' 'services' 'admin.' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']
marital ['married' 'single' 'divorced' 'unknown']
education ['basic.4y' 'high.school' 'basic.6y' 'basic.9y' 'professional.course'
 'unknown' 'university.degree' 'illiterate']
default ['no' 'unknown' 'yes']
housing ['no' 'yes' 'unknown']
loan ['no' 'yes' 'unknown']
contact ['telephone' 'cellular']
month ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'mar' 'apr' 'sep']
day_of_week ['mon' 'tue' 'wed' 'thu' 'fri']
duration [ 261  149  226 ... 1246 1556 1868]
campaign [ 1  2  3  4  5  6  7  8  9 10 11 12 13 19 18 23 14 22 25 16 17 15 20 56
 39 35 42 28 26 27 32 21 24 29 31 30 41 37 40 33 34 43]
pdays [999

We have identified:
-  "unknown" in job
- "unknown" in marital
- "unknown" in education
- "unknown" in default
- "unknown" in housing
- "unknown" in loan
- pdayds = 999 for this dataset represents no contact or unknonw.

In [25]:
#  counting unknowns and 999 in each columns where it is identified
print("job - ",(df["job"] == "unknown").sum())
print("marital -",(df["marital"] == "unknown").sum())
print("education -",(df["education"] == "unknown").sum())
print("default - ",(df["default"] == "unknown").sum())
print("housing - ",(df["housing"] == "unknown").sum())
print("loan - ",(df["loan"] == "unknown").sum())
print("pdays - ",(df["pdays"] == 999).sum())


job -  330
marital - 80
education - 1731
default -  8597
housing -  990
loan -  990
pdays -  39673


In [26]:
(df == "unknown").sum()

,0
age,0
job,330
marital,80
education,1731
default,8597
housing,990
loan,990
contact,0
month,0
day_of_week,0


Two useful distinctions:
- default is the one that really stands out as a substantial "unknown" category.
- education is noticeable, but much smaller.
- The rest are relatively limited.
- 999 is not missingness to “clean away”; it represents a meaningful condition: no prior contact.